In [ ]:
import re
from pathlib import Path
import pandas as pd

# -----------------------------
# 1) File discovery
# -----------------------------
ROUND_FILE_RE = re.compile(
    r"""(?ix)
    ^(?:r|round|brainstormin\ round)?\s*(\d+)\.txt$
    """
)

def find_round_files(folder: str | Path) -> list[tuple[int, Path]]:
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    found = []
    for p in folder.iterdir():
        if not p.is_file():
            continue
        m = ROUND_FILE_RE.match(p.name)
        if m:
            found.append((int(m.group(1)), p))

    if not found:
        raise FileNotFoundError(
            f"No round .txt files found in {folder}. "
            "Expected names like r1.txt, r2.txt, round3.txt, etc."
        )

    return sorted(found, key=lambda x: x[0])


# -----------------------------
# 2) Parsing
# -----------------------------
LINE_RE = re.compile(
    r"""^
    \s*(?P<table>\d*)\s+
    (?P<player>.+?)\s{2,}
    (?P<opponent>.+?)\s{2,}
    (?P<points>\d+\s*-\s*\d+|\d+)\s*
    $""",
    re.VERBOSE,
)

def parse_round_file(path: Path, round_num: int) -> list[dict]:
    rows = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.rstrip("\n")

        # skip junk/header lines
        if not line.strip():
            continue
        if "Table" in line or set(line.strip()) <= {"-"}:
            continue
        if line.strip().lower().startswith("round"):
            continue

        m = LINE_RE.match(line)
        if not m:
            continue

        table = m.group("table").strip() or None
        player = m.group("player").strip()
        opponent = m.group("opponent").strip()
        points = m.group("points").replace(" ", "")

        rows.append(
            {
                "Round": round_num,
                "Table": table,
                "Player": player,
                "Opponent": opponent,
                "PointsRaw": points,
            }
        )
    return rows


def build_matches_df(folder: str | Path) -> pd.DataFrame:
    round_files = find_round_files(folder)

    records = []
    for rnd, path in round_files:
        records.extend(parse_round_file(path, rnd))

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError("Parsed 0 rows. Check file formatting or regex.")

    # numeric parsing:
    # - we want the *cumulative total* for the player after this round.
    # - in your files: "6-3", "9-6", etc. => take the left side (player's total)
    df["PlayerTotal"] = (
        df["PointsRaw"]
        .astype(str)
        .str.extract(r"^\s*(\d+)")
        .astype(int)
    )

    # normalize bye marker
    df["IsBye"] = df["Opponent"].str.strip().eq("***Bye***")

    return df.sort_values(["Round", "Table", "Player"]).reset_index(drop=True)


# -----------------------------
# 3) Derive outcomes by comparing to previous totals
# -----------------------------
def interpret_results(df_matches: pd.DataFrame) -> pd.DataFrame:
    # running totals per player
    prev_totals: dict[str, int] = {}

    out = []
    for rnd in sorted(df_matches["Round"].unique()):
        df_r = df_matches[df_matches["Round"] == rnd]

        for _, row in df_r.iterrows():
            player = row["Player"]
            opponent = row["Opponent"]
            new_total = int(row["PlayerTotal"])
            prev_total = int(prev_totals.get(player, 0))
            gain = new_total - prev_total

            # update running totals (even for byes, so next round math works)
            prev_totals[player] = new_total

            # ignore byes in outcome calculations (but keep row if you want)
            if row["IsBye"]:
                outcome = "Bye"
            else:
                if gain == 3:
                    outcome = "Win"
                elif gain == 1:
                    outcome = "Tie"
                elif gain == 0:
                    outcome = "Loss"
                else:
                    outcome = f"Check({gain})"

            out.append(
                {
                    "Round": rnd,
                    "Table": row["Table"],
                    "Player": player,
                    "Opponent": opponent,
                    "PrevTotal": prev_total,
                    "NewTotal": new_total,
                    "Gain": gain,
                    "Outcome": outcome,
                    "IsBye": bool(row["IsBye"]),
                }
            )

    return pd.DataFrame(out).sort_values(["Round", "Table", "Player"]).reset_index(drop=True)


# -----------------------------
# 4) Optional: turn player-rows into match-level rows (Player A vs Player B)
#    (useful for Elo updates)
# -----------------------------
def build_matchups(df_matches: pd.DataFrame) -> pd.DataFrame:
    """
    Creates one row per actual match (ignores byes), by pairing reciprocal rows
    within the same round (Player vs Opponent).
    """
    df = df_matches[~df_matches["IsBye"]].copy()

    # canonical key so A-vs-B and B-vs-A map together
    def key(a, b, rnd):
        lo, hi = sorted([a.strip(), b.strip()])
        return (rnd, lo, hi)

    df["MatchKey"] = [key(a, b, r) for a, b, r in zip(df["Player"], df["Opponent"], df["Round"])]

    grouped = df.groupby("MatchKey", as_index=False)

    match_rows = []
    for _, g in grouped:
        if len(g) != 2:
            # if something weird happens (missing reciprocal row), keep it visible
            match_rows.append(
                {
                    "Round": int(g["Round"].iloc[0]),
                    "PlayerA": g["Player"].iloc[0],
                    "PlayerB": g["Opponent"].iloc[0],
                    "A_Total": int(g["PlayerTotal"].iloc[0]),
                    "B_Total": None,
                    "Status": f"Unpaired({len(g)})",
                }
            )
            continue

        r0 = g.iloc[0]
        r1 = g.iloc[1]

        # r0 is "Player" vs "Opponent", and r1 should be reverse.
        # Determine winner by comparing *gain* from previous round is better,
        # but we can also just use inferred deltas from totals if you want later.
        match_rows.append(
            {
                "Round": int(r0["Round"]),
                "PlayerA": r0["Player"],
                "PlayerB": r0["Opponent"],
                "A_Total": int(r0["PlayerTotal"]),
                "B_Total": int(r1["PlayerTotal"]),
                "Status": "OK",
            }
        )

    return pd.DataFrame(match_rows).sort_values(["Round", "PlayerA", "PlayerB"]).reset_index(drop=True)


# -----------------------------
# Usage
# -----------------------------
folder = r"2_13_26"  # <-- set this

df_matches = build_matches_df(folder)
df_results = interpret_results(df_matches)
df_matchups = build_matchups(df_matches)

print(f"Parsed rows: {len(df_matches)}")
print(f"Result rows: {len(df_results)}")
print(f"Matchup rows: {len(df_matchups)}")

# If you want to ignore byes completely in df_results:
df_results_no_byes = df_results[~df_results["IsBye"]].copy()

# Save
df_matches.to_csv(Path(folder) / "matches_parsed.csv", index=False)
df_results.to_csv(Path(folder) / "results_interpreted.csv", index=False)
df_matchups.to_csv(Path(folder) / "matchups.csv", index=False)

Parsed rows: 91
Result rows: 91
Matchup rows: 88
